# Day 1: Data preparation and first results

**Paper:** Buying or Generating? Predictors of Renewable Energy Pathways in European SMEs

**Data:** Flash Eurobarometer 549 (GESIS study ZA8869, fieldwork June 2024)

This notebook does four things:

1. Loads the survey and checks, country by country, that each firm's answers hang together.
2. Selects the sample: SMEs in the 20 EU countries that pass that check.
3. Builds the outcome (which energy pathway each firm is on) and the predictors.
4. Produces the first descriptive tables and figures, then saves a clean dataset for Day 2.

**How to run it:** click *Runtime > Run all* in Google Colab. When asked, upload the `.sav` file you downloaded from GESIS. Cells marked **YOUR TURN** are for you to change and re-run.

## 0. Setup

In [ ]:
# Installs the one package Colab does not have by default (it reads SPSS .sav files).
!pip -q install pyreadstat

In [ ]:
import os
import numpy as np
import pandas as pd
import pyreadstat
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

DATA_FILE = "ZA8869_v1-0-0.sav"
if not os.path.exists(DATA_FILE):
    try:
        from google.colab import files           # only exists inside Google Colab
        print("Choose the .sav file you downloaded from GESIS")
        uploaded = files.upload()
        DATA_FILE = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError("Put the .sav file in the same folder as this notebook.")
print("Using:", DATA_FILE)

## 1. Load the survey

`raw` holds the numeric codes. `meta` holds the question wording and what each code means.

In [ ]:
raw, meta = pyreadstat.read_sav(DATA_FILE)
labels = meta.variable_value_labels          # e.g. labels["n1a"] -> {1: "Yes, you purchase...", ...}
print("Firms:", raw.shape[0], "| Variables:", raw.shape[1])
print("Countries:", raw["isocntry"].nunique())

In [ ]:
# A helper to look up any question. Try it with other variable names from the codebook.
def describe(var):
    print(var, "-", meta.column_names_to_labels[var])
    counts = raw[var].value_counts(dropna=False).sort_index()
    for code_, n in counts.items():
        text = labels.get(var, {}).get(code_, "(blank: question not asked)" if pd.isna(code_) else "")
        print(f"   {code_}: {n:>6}  {text}")

describe("n1a")

## 1b. Check that each country's records hang together

Before using the file we test it. A firm with more employees has a higher turnover, so in any genuine business survey the two are strongly related. If they are unrelated inside a country, the turnover on a row does not belong to the firm described on that row, and nothing that links one answer to another can be trusted there.

Two control pairs are included. Size class and number of employees are two answers to the same question, and so are age band and year founded. These should agree everywhere.

In [ ]:
from scipy.stats import spearmanr
chk = raw.copy()
chk["employees"] = chk["scr10a"].where(chk["scr10a"] < 99990)       # the top codes mean "don't know"
chk["turnover"] = chk["scr14"].where(chk["scr14"] < 10)
chk["year"] = chk["scr12a"].where(chk["scr12a"] < 9998)
def rho(g, a, b):
    x = g[[a, b]].dropna()
    return np.nan if len(x) < 30 or x[a].nunique() < 2 or x[b].nunique() < 2 else spearmanr(x[a], x[b])[0]
record_check = pd.DataFrame({c: {"Firms": len(g), "EU27": "yes" if g["eu27"].iloc[0] == 1 else "no",
                                 "Employees vs turnover": rho(g, "employees", "turnover"),
                                 "Control: size class vs employees": rho(g, "scr10", "employees"),
                                 "Control: age band vs year founded": abs(rho(g, "scr12", "year"))} for c, g in chk.groupby("isocntry")}).T
num = ["Employees vs turnover", "Control: size class vs employees", "Control: age band vs year founded"]
record_check[num] = record_check[num].astype(float); record_check["Firms"] = record_check["Firms"].astype(int)
record_check = record_check.sort_values("Employees vs turnover")
THRESHOLD = 0.2
FAILED = sorted(record_check.index[record_check["Employees vs turnover"] < THRESHOLD])
FAILED_EU = sorted(c for c in FAILED if record_check.loc[c, "EU27"] == "yes")
record_check["Passes"] = np.where(record_check["Employees vs turnover"] < THRESHOLD, "no", "yes")
print("Countries that fail (correlation below %.1f):" % THRESHOLD, FAILED)
print("Of these, in the EU27:", FAILED_EU)
print("Lowest correlation among the countries that pass: %.2f" % record_check.loc[record_check["Passes"] == "yes", "Employees vs turnover"].min())
record_check.round(2)

**What to look for.** The countries split into two groups with a wide gap between them: a handful near zero, and the rest at 0.33 or above. The control columns are high everywhere, so answers within one question are intact and it is the link between questions that is broken. We cannot tell from the file whether answers were attached to the wrong firms or the interviews were not genuine. Either way those countries cannot be used to study how one answer relates to another, so they are left out of the sample. `Day6_data_integrity.ipynb` shows what the results look like if they are kept in.

## 2. Select the sample

Four filters, applied in order. Each one is a choice we must report in the paper.

| Step | Rule | Why |
|---|---|---|
| 1 | EU27 countries only | Comparable policy setting; matches the published literature |
| 2 | Fewer than 250 employees | The EU definition of an SME. The survey also interviewed larger firms |
| 3 | A valid answer on renewable energy use | "Don't know" cannot be assigned to a pathway |
| 4 | Country passes the record check | See section 1b |

In [ ]:
flow = []
d = raw.copy()
flow.append(("All firms interviewed", len(d)))

d = d[d["eu27"] == 1]
flow.append(("EU27 only", len(d)))

d = d[d["scr10"].isin([1, 2, 3])]             # 1 = 1-9, 2 = 10-49, 3 = 50-249 employees
flow.append(("SMEs only (1 to 249 employees)", len(d)))

n_dont_know = int((d["n1a"] == 4).sum())
d = d[d["n1a"].isin([1, 2, 3])]
flow.append(("Valid answer on renewable energy use", len(d)))

passes = ~d["isocntry"].isin(FAILED_EU)
flow.append(("Countries that pass the record check (20 of 27)", int(passes.sum())))
# Note: `d` still holds all 27 countries here. The columns are built for all of them first, so that a 27-country
# file can be saved for Day 6, and the seven countries are removed at the end of section 4.

sample_flow = pd.DataFrame(flow, columns=["Step", "Firms remaining"])
sample_flow["Removed at this step"] = -sample_flow["Firms remaining"].diff().fillna(0).astype(int)
print(f"'Don't know' on renewable energy among EU27 SMEs: {n_dont_know}")
print("Firms per country in the final sample: %d to %d" % (d.loc[passes, "isocntry"].value_counts().min(), d.loc[passes, "isocntry"].value_counts().max()))
sample_flow

### Do we reproduce the official figures?

The European Commission published 23% (purchase) and 12% (generate on-site) for EU27 SMEs from this survey. As a check on our filters and weights we compute the same thing on all 27 countries, first as the Commission does (firms answering "don't know" kept in the base) and then our way (those firms removed).

In [ ]:
all27 = raw[(raw["eu27"] == 1) & (raw["scr10"].isin([1, 2, 3]))]
w = all27["w87_sme"]
with_dk = (100 * w.groupby(all27["n1a"]).sum() / w.sum()).rename({1: "Purchased", 2: "On-site", 3: "No renewable", 4: "Don't know"})
valid = all27[all27["n1a"].isin([1, 2, 3])]; w = valid["w87_sme"]
without_dk = (100 * w.groupby(valid["n1a"]).sum() / w.sum()).rename({1: "Purchased", 2: "On-site", 3: "No renewable"})
official_check = pd.DataFrame({"All 27 countries, 'don't know' in the base (as published)": with_dk,
                               "All 27 countries, valid answers only": without_dk}).round(1).loc[["Purchased", "On-site", "No renewable", "Don't know"]]
official_check

## 3. Build the outcome: the energy pathway

Survey question N1a: *Does your company use renewable energy sources?*

| Code in survey | Answer | Our label |
|---|---|---|
| 3 | No | `No renewable` |
| 1 | Yes, purchased from a renewable energy supplier | `Purchased` |
| 2 | Yes, generated on-site (solar panels, wind) | `On-site` |

A limitation to state in the paper: the question allows one answer only, so a firm that both buys and generates is recorded under a single pathway.

In [ ]:
clean = pd.DataFrame(index=d.index)
clean["firm_id"] = d["uniqid"].astype("int64")

pathway_map = {3: "No renewable", 1: "Purchased", 2: "On-site"}
PATHWAYS = ["No renewable", "Purchased", "On-site"]
clean["pathway"] = pd.Categorical(d["n1a"].map(pathway_map), categories=PATHWAYS)
clean["pathway_code"] = clean["pathway"].cat.codes        # 0 = No renewable, 1 = Purchased, 2 = On-site

# Follow-up for on-site generators only: share of energy generated on-site.
share_map = {1: "Less than 5%", 2: "5-19%", 3: "20-49%", 4: "50-74%", 5: "75% or more", 6: "Unknown"}
clean["onsite_share"] = d["n1b"].map(share_map)

# Second outcome, used later for research question 3: plans to use renewable energy within two years.
clean["plans_renewable"] = d["q2.3"].astype(int)

clean["pathway"].value_counts().reindex(PATHWAYS).to_frame("Firms (all 27 countries, before the record check is applied)")

## 4. Build the predictors

We keep two sets, because they answer different questions.

- **Set A, structural:** what the firm *is* (country, sector, size, age, turnover). These cannot be a result of the energy decision, so they are the cleaner test of what drives it.
- **Set B, behavioural:** what the firm *does* (other green actions, investment, barriers, support, climate strategy). These predict better, but they may move together with the energy decision without causing it.

Two rules used throughout:

- **"Don't know" is kept as its own category**, not deleted. Deleting it would drop thousands of firms.
- **A blank means the question was not asked** (for example, barriers were only asked of firms that take at least one action). For yes/no items we set blanks to 0 and keep a flag showing who was asked.

In [ ]:
# ---------- Set A: structural ----------
clean["country"] = d["isocntry"]

region_of = {}
for c in ["AT", "BE", "DE", "FR", "IE", "LU", "NL"]:                              region_of[c] = "Western"
for c in ["DK", "FI", "SE"]:                                                      region_of[c] = "Northern"
for c in ["CY", "ES", "GR", "IT", "MT", "PT"]:                                    region_of[c] = "Southern"
for c in ["BG", "CZ", "EE", "HR", "HU", "LT", "LV", "PL", "RO", "SI", "SK"]:      region_of[c] = "Central and Eastern"
clean["region"] = clean["country"].map(region_of)

clean["sector"] = d["nace_b"].map({1: "Manufacturing", 2: "Industry", 3: "Retail", 4: "Services"})
clean["nace_section"] = d["nace_a"].map(lambda x: labels["nace_a"][x].split(" - ")[0])   # letter B..M
clean["size"] = pd.Categorical(d["scr10"].map({1: "Micro (1-9)", 2: "Small (10-49)", 3: "Medium (50-249)"}),
                               categories=["Micro (1-9)", "Small (10-49)", "Medium (50-249)"])

clean["employees"] = d["scr10a"].where(d["scr10a"] < 999998)          # exact number, blank if not given
year = d["scr12a"].where(d["scr12a"] < 9998)
clean["firm_age_years"] = 2024 - year
clean["age_band"] = d["scr12"].map({1: "Before 2016", 2: "2016-2018", 3: "2019-2022", 4: "2023 or later", 5: "Unknown"})

turnover_labels = {1: "1: under 25k", 2: "2: 25k-50k", 3: "3: 50k-100k", 4: "4: 100k-250k", 5: "5: 250k-500k",
                   6: "6: 500k-2m", 7: "7: 2m-10m", 8: "8: 10m-50m", 9: "9: over 50m", 10: "Unknown"}
clean["turnover_band"] = d["scr14"].map(turnover_labels)
clean["turnover_unknown"] = (d["scr14"] == 10).astype(int)

trend = {1: "Grew 10%+ a year", 2: "Grew under 10% a year", 3: "Unchanged", 4: "Decreased", 998: "Unknown"}
clean["turnover_trend"] = d["scr13a"].map(trend)
clean["employment_trend"] = d["scr11a"].map(trend)
clean["hiring_difficulty"] = d["scr11b"].map({1: "Very difficult", 2: "Slightly difficult", 3: "Not difficult",
                                              4: "Did not hire", 998: "Unknown"})
clean["sells"] = d["scr16"].map({1: "Products", 2: "Services", 3: "Both", 4: "Unknown"})

STRUCTURAL = ["country", "region", "sector", "nace_section", "size", "employees", "firm_age_years", "age_band",
              "turnover_band", "turnover_unknown", "turnover_trend", "employment_trend", "hiring_difficulty", "sells"]
print(len(STRUCTURAL), "structural predictors")

In [ ]:
# ---------- Set B: behavioural ----------
def yes_no(var):
    # 1 if mentioned, 0 if not mentioned or not asked
    return d[var].fillna(0).astype(int)

# Other resource-efficiency actions. q1.3 (renewable energy) is deliberately left out: see the note below.
actions = {"q1.1": "act_save_water", "q1.2": "act_save_energy", "q1.4": "act_save_materials",
           "q1.5": "act_greener_suppliers", "q1.6": "act_minimise_waste", "q1.7": "act_sell_waste",
           "q1.8": "act_recycle", "q1.9": "act_design_for_repair"}
for var, name in actions.items():
    clean[name] = yes_no(var)
clean["n_other_actions"] = clean[list(actions.values())].sum(axis=1)

# Firms that take at least one action were asked the follow-up questions (investment, support, barriers).
clean["asked_followups"] = d["q4"].notna().astype(int)

clean["investment"] = d["q4"].map({1: "0: nothing", 2: "1: under 1% of revenue", 3: "2: 1-5%", 4: "3: 6-10%",
                                   5: "4: 11-30%", 6: "5: over 30%", 7: "Unknown"}).fillna("No actions taken")

support = {"q5.1": "sup_own_finance", "q5.2": "sup_own_expertise", "q5.3": "sup_external",
           "q6.1": "ext_public_funding", "q6.2": "ext_private_funding", "q6.3": "ext_friends_family",
           "q6.4": "ext_advice_public", "q6.5": "ext_advice_consultants", "q6.6": "ext_advice_associations",
           "q6.7": "ext_advice_supply_chain"}
barriers = {"q7.1": "bar_admin_complexity", "q7.2": "bar_adapting_legislation", "q7.3": "bar_outdated_tech_rules",
            "q7.4": "bar_choosing_actions", "q7.5": "bar_cost", "q7.6": "bar_lack_expertise",
            "q7.7": "bar_lack_supply", "q7.8": "bar_lack_demand", "q7.9": "bar_labelling_complexity",
            "q7.10": "bar_reporting_requirements", "q7.12": "bar_none"}
would_help = {"q8.1": "help_self_assessment_tool", "q8.2": "help_consultancy", "q8.3": "help_grants",
              "q8.4": "help_funding_advice", "q8.5": "help_tech_demonstration", "q8.6": "help_case_studies",
              "q8.7": "help_cross_sector_cooperation", "q8.8": "help_clearer_rules"}
for group in (support, barriers, would_help):
    for var, name in group.items():
        clean[name] = yes_no(var)
clean["n_barriers"] = clean[[v for v in barriers.values() if v != "bar_none"]].sum(axis=1)

clean["climate_strategy"] = d["q14"].map({1: "Has a strategy", 2: "Planning one", 3: "No plans",
                                          4: "Already climate neutral", 5: "Unknown"})
clean["green_products"] = d["q9"].map({1: "Yes", 2: "Planning", 3: "No", 4: "Unknown"})
clean["green_jobs"] = d["dx5r"].map({1: "0", 2: "1-5", 3: "6-9", 4: "10-50", 5: "51-100", 6: "101+", 7: "Unknown"})
clean["recycled_materials"] = d["n2"].map({1: "0%", 2: "Under 5%", 3: "5-19%", 4: "20-49%", 5: "50-74%",
                                           6: "75%+", 7: "Unknown"}).fillna("Does not sell products")
clean["lifespan_policy"] = d["n3"].map({1: "All products", 2: "Some products", 3: "No",
                                        4: "Unknown"}).fillna("Does not sell products")

BEHAVIOURAL = (list(actions.values()) + ["n_other_actions", "asked_followups", "investment"] + list(support.values())
               + list(barriers.values()) + ["n_barriers"] + list(would_help.values())
               + ["climate_strategy", "green_products", "green_jobs", "recycled_materials", "lifespan_policy"])
print(len(BEHAVIOURAL), "behavioural predictors")

# Weights supplied with the survey (they correct for which firms were easier to reach).
clean["weight_eu27"] = d["w87_sme"]        # use for figures about SMEs in the 20 countries as a whole
clean["weight_country"] = d["w1_sme"]      # use for figures within one country

# Now apply the fourth filter: keep the 20 countries that pass the record check.
clean["records_ok"] = (~clean["country"].isin(FAILED_EU)).astype(int)
clean27 = clean.copy()                                         # all 27 countries, for Day 6 only
clean = clean[clean["records_ok"] == 1].drop(columns="records_ok")
d = d.loc[clean.index]
print("Final sample:", len(clean), "firms in", clean["country"].nunique(), "countries  (27-country file:", len(clean27), "firms)")
clean["pathway"].value_counts().reindex(PATHWAYS).to_frame("Firms in the final sample")

### Why q1.3 is excluded

Question Q1.3 asks whether the firm is *"using predominantly renewable energy"*. That is almost the same thing as our outcome, so a model given this variable would look accurate without having learned anything. This mistake is called **data leakage**. The table below shows how closely the two overlap.

In [ ]:
leak = pd.crosstab(d["q1.3"].map({0: "Q1.3 not mentioned", 1: "Q1.3 mentioned"}), clean["pathway"])
leak["% using renewables"] = (100 * (leak["Purchased"] + leak["On-site"]) / leak.sum(axis=1)).round(1)
leak

## 5. How much is unknown or not asked?

Reported openly in the paper, since it affects which firms the models learn from.

In [ ]:
rows = []
for col in STRUCTURAL + BEHAVIOURAL:
    s = clean[col]
    unknown = 0 if pd.api.types.is_numeric_dtype(s) else int((s.astype(str) == "Unknown").sum())
    rows.append((col, int(s.isna().sum()), unknown))
missing = pd.DataFrame(rows, columns=["Variable", "Blank", "Answered 'Unknown'"])
missing["% of sample"] = (100 * (missing["Blank"] + missing["Answered 'Unknown'"]) / len(clean)).round(1)
missing = missing[missing["% of sample"] > 0].sort_values("% of sample", ascending=False).reset_index(drop=True)
missing

## 6. First results

Two versions of every percentage:

- **Unweighted:** the raw share among firms interviewed. This is what the machine learning models will see.
- **Weighted:** corrected to represent all SMEs in the 20 countries. This is what we report as a fact about those countries.

They differ because the survey interviewed proportionally more medium-sized firms and firms from small countries than exist in reality.

In [ ]:
def pathway_table(by=None, weight=None):
    # Percentage of firms on each pathway, overall or split by a column of `clean`.
    w = clean[weight] if weight else pd.Series(1.0, index=clean.index)
    if by is None:
        t = (w.groupby(clean["pathway"], observed=False).sum() / w.sum() * 100).to_frame().T
        t.index = ["All SMEs"]
    else:
        t = w.groupby([clean[by], clean["pathway"]], observed=False).sum().unstack()
        t = t.div(t.sum(axis=1), axis=0) * 100
    t = t[PATHWAYS]
    t["Any renewable"] = t["Purchased"] + t["On-site"]          # added before rounding, so the total is not rounded twice
    return t.round(1)

overall = pd.concat([pathway_table().rename(index={"All SMEs": "Unweighted"}),
                     pathway_table(weight="weight_eu27").rename(index={"All SMEs": "Weighted"})])
overall.insert(0, "Firms", [len(clean), len(clean)])
overall

In [ ]:
def with_n(by, weight):
    t = pathway_table(by, weight)
    t.insert(0, "Firms", clean.groupby(by, observed=False).size())
    return t

by_size    = with_n("size", "weight_eu27")
by_sector  = with_n("sector", "weight_eu27").sort_values("Any renewable", ascending=False)
by_region  = with_n("region", "weight_eu27").sort_values("Any renewable", ascending=False)
by_country = with_n("country", "weight_country").sort_values("Any renewable", ascending=False)

print("Weighted % of SMEs on each pathway\n")
for name, t in [("By size", by_size), ("By sector", by_sector), ("By region", by_region)]:
    print(name); display(t)

# Among firms that generate on-site: what share of their energy does it cover? (survey item N1b)
on = clean[clean["pathway"] == "On-site"]
order = ["Less than 5%", "5-19%", "20-49%", "50-74%", "75% or more", "Unknown"]
onsite_share = pd.DataFrame({"Firms": on["onsite_share"].value_counts().reindex(order),
                             "Weighted %": (100 * on.groupby("onsite_share")["weight_eu27"].sum() / on["weight_eu27"].sum()).reindex(order).round(1)})
print("Share of energy generated on-site, among on-site generators"); display(onsite_share)
# Across countries, do purchasing and generating go together?
wc_ = clean["weight_country"]
exact = wc_.groupby([clean["country"], clean["pathway"]], observed=False).sum().unstack().pipe(lambda t: t.div(t.sum(axis=1), axis=0))   # not rounded
rho_cc, p_cc = spearmanr(exact["Purchased"], exact["On-site"])
print("Across the %d countries, share purchasing vs share generating: Spearman rho = %.2f (p = %.2f) weighted, %.2f unweighted" % (
    len(by_country), rho_cc, p_cc,
    spearmanr(*[pd.crosstab(clean["country"], clean["pathway"], normalize="index")[k] for k in ("Purchased", "On-site")])[0]))

In [ ]:
by_country

### Are these differences statistically significant?

A chi-square test on the unweighted counts, with Cramér's V as the effect size (0.1 is small, 0.3 medium, 0.5 large). With nearly 10,000 firms nearly everything is significant, so the effect size is the number to read.

In [ ]:
def chi_square(by):
    table = pd.crosstab(clean[by], clean["pathway"])
    chi2, p, dof, _ = chi2_contingency(table)
    v = np.sqrt(chi2 / (table.values.sum() * (min(table.shape) - 1)))
    return {"Variable": by, "Chi-square": round(float(chi2), 1), "df": int(dof), "p-value": f"{p:.2e}",
            "Cramér's V": round(float(v), 3)}

tests = pd.DataFrame([chi_square(v) for v in
                      ["country", "region", "size", "sector", "turnover_band", "climate_strategy", "investment"]])
tests.sort_values("Cramér's V", ascending=False).reset_index(drop=True)

### Figure 1: renewable energy pathways by country

In [ ]:
BLUE, ORANGE, INK, MUTED, GRID = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9, "axes.edgecolor": GRID,
                     "text.color": INK, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": INK})

c = by_country.sort_values("Any renewable")                 # smallest at the bottom
fig, ax = plt.subplots(figsize=(7, 7.6))
ax.barh(c.index, c["Purchased"], color=BLUE, height=0.62, edgecolor="white", linewidth=1.2, label="Purchased from a supplier")
ax.barh(c.index, c["On-site"], left=c["Purchased"], color=ORANGE, height=0.62, edgecolor="white", linewidth=1.2,
        label="Generated on-site")
for y, total in enumerate(c["Any renewable"]):
    ax.text(total + 1, y, f"{total:.0f}%", va="center", fontsize=8, color=MUTED)
ax.set_xlim(0, max(c["Any renewable"]) + 8)
ax.set_xlabel("% of SMEs in the country (weighted)")
ax.xaxis.grid(True, color=GRID, linewidth=0.8); ax.set_axisbelow(True)
for side in ["top", "right", "left"]: ax.spines[side].set_visible(False)
ax.tick_params(length=0)
ax.legend(frameon=False, loc="lower right")
ax.set_title("SMEs using renewable energy, by pathway and country (2024)", loc="left", fontsize=11, fontweight="bold")
plt.tight_layout(); plt.savefig("fig1_pathways_by_country.png", dpi=300); plt.show()

### Figure 2: pathways by firm size and by sector

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8.2, 3.3), sharey=True)
for ax, (t, title) in zip(axes, [(by_size, "By firm size"), (by_sector, "By sector")]):
    x = np.arange(len(t))
    ax.bar(x - 0.19, t["Purchased"], width=0.36, color=BLUE, label="Purchased from a supplier")
    ax.bar(x + 0.19, t["On-site"], width=0.36, color=ORANGE, label="Generated on-site")
    for xi, (p, o) in enumerate(zip(t["Purchased"], t["On-site"])):
        ax.text(xi - 0.19, p + 0.6, f"{p:.0f}", ha="center", fontsize=8, color=MUTED)
        ax.text(xi + 0.19, o + 0.6, f"{o:.0f}", ha="center", fontsize=8, color=MUTED)
    ax.set_xticks(x); ax.set_xticklabels([str(i).replace(" (", "\n(") for i in t.index])
    ax.set_title(title, loc="left", fontsize=10, fontweight="bold")
    ax.yaxis.grid(True, color=GRID, linewidth=0.8); ax.set_axisbelow(True)
    for side in ["top", "right", "left"]: ax.spines[side].set_visible(False)
    ax.tick_params(length=0)
axes[0].set_ylabel("% of SMEs (weighted)")
axes[0].set_ylim(0, max(by_size[["Purchased", "On-site"]].max().max(), by_sector[["Purchased", "On-site"]].max().max()) + 5)
axes[1].legend(frameon=False, loc="upper right", fontsize=8)
plt.tight_layout(); plt.savefig("fig2_pathways_by_size_sector.png", dpi=300); plt.show()

## 7. YOUR TURN

Change `BREAKDOWN` to any column of the clean dataset and re-run the cell. Good ones to try: `"turnover_band"`, `"age_band"`, `"climate_strategy"`, `"investment"`, `"hiring_difficulty"`, `"nace_section"`.

For each one, write down one sentence on what you see. Those sentences become your results section.

In [ ]:
BREAKDOWN = "turnover_band"        # <-- change this and re-run

display(with_n(BREAKDOWN, "weight_eu27"))
print(chi_square(BREAKDOWN))

## 8. Save the outputs

- `sme_energy_clean.csv`: the clean dataset for the 20 countries, one row per firm. Opens in Excel. Day 2 starts from this file.
- `sme_energy_27countries.csv`: the same columns for all 27 countries, with a column `records_ok` marking the 20. Only Day 6 uses it.
- `data_dictionary.csv`: every column, where it came from, and which predictor set it belongs to.
- `Day1_results.xlsx`: all the tables above, one per sheet.

In [ ]:
clean.to_csv("sme_energy_clean.csv", index=False)
clean27.to_csv("sme_energy_27countries.csv", index=False)

source = {"firm_id": "uniqid", "pathway": "n1a", "pathway_code": "n1a", "onsite_share": "n1b", "plans_renewable": "q2.3",
          "country": "isocntry", "region": "isocntry (grouped)", "sector": "nace_b", "nace_section": "nace_a",
          "size": "scr10", "employees": "scr10a", "firm_age_years": "scr12a", "age_band": "scr12",
          "turnover_band": "scr14", "turnover_unknown": "scr14", "turnover_trend": "scr13a",
          "employment_trend": "scr11a", "hiring_difficulty": "scr11b", "sells": "scr16",
          "n_other_actions": "q1.1-q1.9 except q1.3 (count)", "asked_followups": "q4 (asked or not)",
          "investment": "q4", "n_barriers": "q7.1-q7.10 (count)", "climate_strategy": "q14", "green_products": "q9",
          "green_jobs": "dx5r", "recycled_materials": "n2", "lifespan_policy": "n3",
          "weight_eu27": "w87_sme", "weight_country": "w1_sme"}
for group in (actions, support, barriers, would_help):
    source.update({name: var for var, name in group.items()})

def role(col):
    if col in STRUCTURAL: return "Predictor: set A (structural)"
    if col in BEHAVIOURAL: return "Predictor: set B (behavioural)"
    if col in ("pathway", "pathway_code"): return "Outcome (research questions 1 and 2)"
    if col == "plans_renewable": return "Outcome (research question 3)"
    if col == "onsite_share": return "Descriptive only (on-site firms)"
    if col.startswith("weight"): return "Survey weight"
    return "Identifier"

def wording(col):
    first = source[col].split(" ")[0].split("-")[0]
    text = meta.column_names_to_labels.get(first, "")
    item = labels.get(first, {}).get(1.0, "") if first[:2] in ("q1", "q5", "q6", "q7", "q8") else ""
    return f"{text} [{item}]" if item else text

dictionary = pd.DataFrame({"Column": clean.columns})
dictionary["Survey variable"] = dictionary["Column"].map(source)
dictionary["Role"] = dictionary["Column"].map(role)
dictionary["Survey wording"] = dictionary["Column"].map(wording)
dictionary.to_csv("data_dictionary.csv", index=False)

with pd.ExcelWriter("Day1_results.xlsx") as xl:
    sample_flow.to_excel(xl, sheet_name="Sample flow", index=False)
    record_check.round(6).to_excel(xl, sheet_name="Record check")
    pd.DataFrame({"Item": ["Countries", "Share purchasing vs share generating across countries: rho", "same: p-value"],
                  "Value": [len(by_country), rho_cc, p_cc]}).to_excel(xl, sheet_name="Key numbers", index=False)
    official_check.to_excel(xl, sheet_name="Official figures check")
    onsite_share.to_excel(xl, sheet_name="On-site share")
    leak.to_excel(xl, sheet_name="Q1.3 overlap")
    overall.to_excel(xl, sheet_name="Overall")
    by_size.to_excel(xl, sheet_name="By size")
    by_sector.to_excel(xl, sheet_name="By sector")
    by_region.to_excel(xl, sheet_name="By region")
    by_country.to_excel(xl, sheet_name="By country")
    tests.to_excel(xl, sheet_name="Chi-square tests", index=False)
    missing.to_excel(xl, sheet_name="Unknown or blank", index=False)
    dictionary.to_excel(xl, sheet_name="Data dictionary", index=False)

print("Saved. Clean dataset:", clean.shape[0], "firms,", clean.shape[1], "columns")
print("Set A predictors:", len(STRUCTURAL), "| Set B predictors:", len(BEHAVIOURAL))

In [ ]:
# In Google Colab, this downloads the output files to your computer.
try:
    from google.colab import files
    for f in ["sme_energy_clean.csv", "sme_energy_27countries.csv", "data_dictionary.csv", "Day1_results.xlsx",
              "fig1_pathways_by_country.png", "fig2_pathways_by_size_sector.png"]:
        files.download(f)
except ImportError:
    pass